In [2]:
import pandas as pd
import math

# ---------------- Read Dataset ----------------
df = pd.read_csv(r"C:\Users\kodeb\Downloads\electronics..csv")

# Remove RID column if present
if "RID" in df.columns:
    df = df.drop("RID", axis=1)

target = df.columns[-1]

# ---------------- Entropy ----------------
def entropy(data):

    total = len(data)

    counts = {}

    for value in data[target]:
        counts[value] = counts.get(value, 0) + 1

    ent = 0

    for count in counts.values():
        p = count / total
        ent -= p * math.log2(p)

    return ent


# ---------------- Information Gain ----------------
def information_gain(data, attribute):

    total_entropy = entropy(data)

    weighted_entropy = 0

    for value in data[attribute].unique():

        subset = data[data[attribute] == value]

        weight = len(subset) / len(data)

        weighted_entropy += weight * entropy(subset)

    return total_entropy - weighted_entropy


# ---------------- ID3 Algorithm ----------------
def id3(data, attributes, level=0):

    # All records belong to one class
    if len(data[target].unique()) == 1:

        print("   " * level + "Leaf Node -->", data[target].iloc[0])
        return data[target].iloc[0]

    # No attributes left
    if len(attributes) == 0:

        leaf = data[target].mode()[0]
        print("   " * level + "Leaf Node -->", leaf)
        return leaf

    print("\n")
    print("   " * level + "Current Dataset Size :", len(data))
    print("   " * level + "Calculating Information Gain")

    best_attribute = None
    best_gain = -1

    # Calculate Information Gain of every attribute
    for attr in attributes:

        gain = information_gain(data, attr)

        print("   " * level + attr, "=", round(gain, 4))

        if gain > best_gain:
            best_gain = gain
            best_attribute = attr

    print("   " * level + "\nBest Attribute :", best_attribute)
    print("   " * level + "Information Gain :", round(best_gain, 4))

    tree = {best_attribute: {}}

    remaining = []

    for attr in attributes:
        if attr != best_attribute:
            remaining.append(attr)

    # Split dataset
    for value in data[best_attribute].unique():

        print("\n")
        print("   " * level + "Splitting", best_attribute, "=", value)

        subset = data[data[best_attribute] == value]

        tree[best_attribute][value] = id3(subset, remaining, level + 1)

    return tree


# ---------------- Run ----------------
attributes = list(df.columns[:-1])

tree = id3(df, attributes)

print("\n\n==============================")
print("FINAL DECISION TREE")
print("==============================")

print(tree)



Current Dataset Size : 14
Calculating Information Gain
age = 0.2467
income = 0.0292
student = 0.1518
credit_rating = 0.0481

Best Attribute : age
Information Gain : 0.2467


Splitting age = youth


   Current Dataset Size : 5
   Calculating Information Gain
   income = 0.571
   student = 0.971
   credit_rating = 0.02
   
Best Attribute : student
   Information Gain : 0.971


   Splitting student = no
      Leaf Node --> no


   Splitting student = yes
      Leaf Node --> yes


Splitting age = middle_aged
   Leaf Node --> yes


Splitting age = senior


   Current Dataset Size : 5
   Calculating Information Gain
   income = 0.02
   student = 0.02
   credit_rating = 0.971
   
Best Attribute : credit_rating
   Information Gain : 0.971


   Splitting credit_rating = fair
      Leaf Node --> yes


   Splitting credit_rating = excellent
      Leaf Node --> no


FINAL DECISION TREE
{'age': {'youth': {'student': {'no': 'no', 'yes': 'yes'}}, 'middle_aged': 'yes', 'senior': {'credit_rating': {'

In [3]:
import pandas as pd

# ---------------- Read Dataset ----------------
df = pd.read_csv(r"C:\Users\kodeb\Downloads\electronics..csv")

# Remove RID column if present
if "RID" in df.columns:
    df = df.drop("RID", axis=1)

target = df.columns[-1]

# ---------------- Gini ----------------
def gini(data):

    total = len(data)

    counts = {}

    for value in data[target]:
        counts[value] = counts.get(value, 0) + 1

    g = 1

    for count in counts.values():
        p = count / total
        g = g - (p * p)

    return g


# ---------------- Gini Index ----------------
def gini_index(data, attribute):

    total = len(data)

    gini_value = 0

    for value in data[attribute].unique():

        subset = data[data[attribute] == value]

        weight = len(subset) / total

        gini_value += weight * gini(subset)

    return gini_value


# ---------------- CART ----------------
def cart(data, attributes, level=0):

    # All records belong to same class
    if len(data[target].unique()) == 1:

        print("   " * level + "Leaf Node -->", data[target].iloc[0])
        return data[target].iloc[0]

    # No attributes left
    if len(attributes) == 0:

        leaf = data[target].mode()[0]
        print("   " * level + "Leaf Node -->", leaf)
        return leaf

    print("\n")
    print("   " * level + "Current Dataset Size :", len(data))
    print("   " * level + "Calculating Gini Index")

    best_attribute = None
    lowest_gini = 999

    # Calculate Gini Index
    for attr in attributes:

        g = gini_index(data, attr)

        print("   " * level + attr, "=", round(g, 4))

        if g < lowest_gini:
            lowest_gini = g
            best_attribute = attr

    print("\n" + "   " * level + "Best Attribute :", best_attribute)
    print("   " * level + "Lowest Gini Index :", round(lowest_gini, 4))

    tree = {best_attribute: {}}

    remaining = []

    for attr in attributes:
        if attr != best_attribute:
            remaining.append(attr)

    # Split dataset
    for value in data[best_attribute].unique():

        print("\n")
        print("   " * level + "Splitting", best_attribute, "=", value)

        subset = data[data[best_attribute] == value]

        tree[best_attribute][value] = cart(subset, remaining, level + 1)

    return tree


# ---------------- Run ----------------
attributes = list(df.columns[:-1])

tree = cart(df, attributes)

print("\n==============================")
print("FINAL DECISION TREE")
print("==============================")
print(tree)



Current Dataset Size : 14
Calculating Gini Index
age = 0.3429
income = 0.4405
student = 0.3673
credit_rating = 0.4286

Best Attribute : age
Lowest Gini Index : 0.3429


Splitting age = youth


   Current Dataset Size : 5
   Calculating Gini Index
   income = 0.2
   student = 0.0
   credit_rating = 0.4667

   Best Attribute : student
   Lowest Gini Index : 0.0


   Splitting student = no
      Leaf Node --> no


   Splitting student = yes
      Leaf Node --> yes


Splitting age = middle_aged
   Leaf Node --> yes


Splitting age = senior


   Current Dataset Size : 5
   Calculating Gini Index
   income = 0.4667
   student = 0.4667
   credit_rating = 0.0

   Best Attribute : credit_rating
   Lowest Gini Index : 0.0


   Splitting credit_rating = fair
      Leaf Node --> yes


   Splitting credit_rating = excellent
      Leaf Node --> no

FINAL DECISION TREE
{'age': {'youth': {'student': {'no': 'no', 'yes': 'yes'}}, 'middle_aged': 'yes', 'senior': {'credit_rating': {'fair': 'yes', 'excel

In [4]:
import pandas as pd
import math

# ---------------- Read Dataset ----------------
df = pd.read_csv(r"C:\Users\kodeb\Downloads\electronics..csv")

# Remove RollNo column if present
if "RID" in df.columns:
    df = df.drop("RID", axis=1)

target = df.columns[-1]

# ---------------- Entropy ----------------
def entropy(data):

    total = len(data)

    counts = {}

    for value in data[target]:
        counts[value] = counts.get(value, 0) + 1

    ent = 0

    for count in counts.values():
        p = count / total
        ent -= p * math.log2(p)

    return ent


# ---------------- Information Gain ----------------
def information_gain(data, attribute):

    total_entropy = entropy(data)

    weighted_entropy = 0

    for value in data[attribute].unique():

        subset = data[data[attribute] == value]

        weight = len(subset) / len(data)

        weighted_entropy += weight * entropy(subset)

    return total_entropy - weighted_entropy


# ---------------- Split Information ----------------
def split_info(data, attribute):

    total = len(data)

    split = 0

    for value in data[attribute].unique():

        subset = data[data[attribute] == value]

        weight = len(subset) / total

        if weight != 0:
            split -= weight * math.log2(weight)

    return split


# ---------------- Gain Ratio ----------------
def gain_ratio(data, attribute):

    gain = information_gain(data, attribute)

    split = split_info(data, attribute)

    if split == 0:
        return 0

    return gain / split


# ---------------- C4.5 Algorithm ----------------
def c45(data, attributes, level=0):

    # If all records belong to one class
    if len(data[target].unique()) == 1:

        print("   " * level + "Leaf Node -->", data[target].iloc[0])
        return data[target].iloc[0]

    # If no attributes left
    if len(attributes) == 0:

        leaf = data[target].mode()[0]

        print("   " * level + "Leaf Node -->", leaf)

        return leaf

    print("\n")
    print("   " * level + "Current Dataset Size :", len(data))
    print("   " * level + "Calculating Gain Ratio")

    best_attribute = None
    best_ratio = -1

    # Calculate Gain Ratio
    for attr in attributes:

        ratio = gain_ratio(data, attr)

        print("   " * level + attr, "=", round(ratio, 4))

        if ratio > best_ratio:

            best_ratio = ratio

            best_attribute = attr

    print("\n" + "   " * level + "Best Attribute :", best_attribute)
    print("   " * level + "Highest Gain Ratio :", round(best_ratio, 4))

    tree = {best_attribute: {}}

    remaining = []

    for attr in attributes:

        if attr != best_attribute:
            remaining.append(attr)

    # Split Dataset
    for value in data[best_attribute].unique():

        print("\n")
        print("   " * level + "Splitting", best_attribute, "=", value)

        subset = data[data[best_attribute] == value]

        tree[best_attribute][value] = c45(subset, remaining, level + 1)

    return tree


# ---------------- Run ----------------
attributes = list(df.columns[:-1])

tree = c45(df, attributes)

print("\n==============================")
print("FINAL DECISION TREE")
print("==============================")

print(tree)



Current Dataset Size : 14
Calculating Gain Ratio
age = 0.1564
income = 0.0188
student = 0.1518
credit_rating = 0.0488

Best Attribute : age
Highest Gain Ratio : 0.1564


Splitting age = youth


   Current Dataset Size : 5
   Calculating Gain Ratio
   income = 0.3751
   student = 1.0
   credit_rating = 0.0206

   Best Attribute : student
   Highest Gain Ratio : 1.0


   Splitting student = no
      Leaf Node --> no


   Splitting student = yes
      Leaf Node --> yes


Splitting age = middle_aged
   Leaf Node --> yes


Splitting age = senior


   Current Dataset Size : 5
   Calculating Gain Ratio
   income = 0.0206
   student = 0.0206
   credit_rating = 1.0

   Best Attribute : credit_rating
   Highest Gain Ratio : 1.0


   Splitting credit_rating = fair
      Leaf Node --> yes


   Splitting credit_rating = excellent
      Leaf Node --> no

FINAL DECISION TREE
{'age': {'youth': {'student': {'no': 'no', 'yes': 'yes'}}, 'middle_aged': 'yes', 'senior': {'credit_rating': {'fair': 'yes', 